# Stage 04 & 05: NLP Sentiment Analysis & Machine Learning Classification
**Project**: Steam Game Intelligence  
**Notebook**: `notebooks/03_nlp_and_machine_learning.ipynb`  
**Objective**: Perform NLP sentiment theme analysis and build a reproducible ML recommendation prediction pipeline while strictly auditing against target leakage.

---
## Master ML Guidelines & Auditing:
1. **Target**: `is_recommended` (1 = Recommended, 0 = Not Recommended).
2. **Leakage Audit**: All text n-grams and engagement features are extracted strictly at the point of review submission. No future metrics (such as aggregate game sales or post-hoc ratings) are used in prediction.
3. **Pipeline Safety**: Preprocessing and TF-IDF vectorization fit strictly on `X_train` to prevent train-test data leakage.
4. **Metrics**: Evaluated using Precision, Recall, F1-Score, and ROC-AUC (not raw accuracy alone).


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
)

sns.set_theme(style='whitegrid')

# Load Processed Reviews
rev_path = '../data/processed/steam_game_reviews_clean.csv' if os.path.exists('../data/processed/steam_game_reviews_clean.csv') else 'data/processed/steam_game_reviews_clean.csv'
df_reviews = pd.read_csv(rev_path, usecols=['game_name', 'review', 'is_recommended'])
df_reviews['review'] = df_reviews['review'].fillna('')
df_reviews = df_reviews[df_reviews['review'].str.strip() != ''].copy()

print(f"Total valid review records: {len(df_reviews)}")
print("Class Distribution:")
print(df_reviews['is_recommended'].value_counts(normalize=True))


### 1. Model Training & Pipeline Setup
We sample 100,000 reviews for reproducible training, split 80/20 train/test, and fit a TF-IDF + LogisticRegression pipeline.


In [ ]:
df_sample = df_reviews.sample(n=100000, random_state=42).reset_index(drop=True)
X = df_sample['review']
y = df_sample['is_recommended']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Fit Logistic Regression Pipeline
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000, C=1.0, random_state=42))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

print("=== Logistic Regression Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Not Recommended', 'Recommended']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")


### 2. Feature Importance & Sentiment Keyword Inspection
Inspecting the top positive and negative n-gram weights learned by the model.


In [ ]:
tfidf = pipe.named_steps['tfidf']
clf = pipe.named_steps['clf']
feature_names = np.array(tfidf.get_feature_names_out())
coefs = clf.coef_[0]

top_pos_idx = np.argsort(coefs)[-15:]
top_neg_idx = np.argsort(coefs)[:15]

df_pos = pd.DataFrame({'ngram': feature_names[top_pos_idx], 'coefficient': coefs[top_pos_idx]})
df_neg = pd.DataFrame({'ngram': feature_names[top_neg_idx], 'coefficient': coefs[top_neg_idx]})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=df_pos, x='coefficient', y='ngram', ax=axes[0], hue='ngram', palette='Greens_r', legend=False)
axes[0].set_title('Top Positive Sentiment Predictors')

sns.barplot(data=df_neg, x='coefficient', y='ngram', ax=axes[1], hue='ngram', palette='Reds', legend=False)
axes[1].set_title('Top Negative Sentiment Predictors')

plt.tight_layout()
plt.show()


### 3. Model Checkpoint & Persistence
Saved trained pipeline to `models/recommendation_pipeline.joblib`.
